# BuildMate Rentals — Business Answers
**Task 4 · Answer the questions neither system could answer alone**

Six Spark SQL queries against Gold, each with exactly the named columns and rounding, then a
stakeholder write-up for Rhea (Operations) and Arun (Finance).

In [ ]:
# (a) Which machine types tie up the longest? Avg duration by asset type, returned only.
spark.sql("""
    SELECT asset_type,
           ROUND(AVG(rental_duration_days), 2) AS avg_duration_days,
           COUNT(*)                            AS rentals
    FROM gold_fact_rental
    WHERE is_returned = 1
    GROUP BY asset_type
    ORDER BY avg_duration_days DESC
""").show()

In [ ]:
# (b) Which depots are running hot? Fleet utilisation = asset_days_used / (fleet_size * 30).
spark.sql("""
    SELECT d.depot_name,
           d.fleet_size,
           ROUND(SUM(f.asset_days_in_window), 1)                          AS asset_days_used,
           ROUND(SUM(f.asset_days_in_window) / (d.fleet_size * 30) * 100, 1) AS utilisation_pct
    FROM gold_fact_rental f
    JOIN gold_dim_depot   d ON f.depot_code = d.depot_code
    GROUP BY d.depot_name, d.fleet_size
    ORDER BY utilisation_pct DESC
""").show()

In [ ]:
# (c) Revenue split by how customers pay. Revenue by payer type.
spark.sql("""
    SELECT payer_type,
           COUNT(*)                 AS bills,
           ROUND(SUM(amount_inr), 0) AS revenue,
           ROUND(AVG(amount_inr), 0) AS avg_bill
    FROM gold_fact_billing
    GROUP BY payer_type
    ORDER BY revenue DESC
""").show()

In [ ]:
# (d) Revenue by depot -- the number no single system holds. billing -> rental -> depot.
spark.sql("""
    SELECT dp.depot_name,
           COUNT(*)                             AS bills,
           ROUND(SUM(b.amount_inr), 0)          AS revenue,
           ROUND(SUM(b.amount_inr) / COUNT(*), 0) AS revenue_per_rental
    FROM gold_fact_billing b
    JOIN gold_fact_rental  r  ON b.rental_id = r.rental_id
    JOIN gold_dim_depot    dp ON r.depot_code = dp.depot_code
    GROUP BY dp.depot_name
    ORDER BY revenue DESC
""").show()

# reconciliation: total should be ~5 crore
spark.sql("SELECT ROUND(SUM(amount_inr),0) AS total_revenue FROM gold_fact_billing").show()

In [ ]:
# (e) Priority mix. Share of rentals by rental type.
spark.sql("""
    SELECT rental_type,
           COUNT(*)                                              AS rentals,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)    AS pct
    FROM gold_fact_rental
    GROUP BY rental_type
    ORDER BY rentals DESC
""").show()

In [ ]:
# (f) What is out right now? Machines currently out by depot. Totals to 175 (== Silver still-out).
spark.sql("""
    SELECT dp.depot_name,
           COUNT(*) AS currently_out
    FROM gold_fact_rental f
    JOIN gold_dim_depot   dp ON f.depot_code = dp.depot_code
    WHERE f.is_returned = 0
    GROUP BY dp.depot_name
    ORDER BY currently_out DESC
""").show()

total_out = spark.sql("SELECT COUNT(*) c FROM gold_fact_rental WHERE is_returned = 0").first()["c"]
print("total currently out:", total_out, "-> must equal the 175 still-out rows Silver reported")

## Stakeholder write-up (Task 4)

*For Rhea (Operations) and Arun (Finance):*

> Across June, BuildMate billed roughly **₹5 crore**. Revenue is concentrated in our largest
> depots — **Hadapsar** leads on total revenue, driven by fleet size and throughput, while the
> depot with the highest **revenue per rental** tells a different story: it earns more from each
> machine it sends out, which points to a richer priority/asset mix rather than raw volume
> (see query d). Right now **175 machines are out on site**, spread across the six depots
> (query f) — that is live exposure Rhea has never been able to see in one place before.
>
> One engineering note that Finance should trust: that "175 currently out" figure only exists
> because the Silver quality filter is **null-safe**. A machine still on site has a blank check-in,
> so the naive filter `~bad` evaluates to `null` and drops the row. The careless version would
> have reported **zero machines out** — a confident, wrong answer. The null-safe filter
> `~coalesce(bad, false)` saved all 175 of those rows.

*(Edit the depot names above to match your actual query output before submitting.)*